In [ ]:
# import libraries
import pandas as pd
import pandas as pd
import spacy
from spacy.language import Language
from spacy_langdetect import LanguageDetector
import re
import string

In [ ]:
# load the non-English projects dataset
non_english_df = pd.read_csv('non_english_projects.csv')
print("Non-English projects DataFrame shape:", non_english_df.shape)

Non-English projects DataFrame shape: (3874, 78)


In [ ]:
# set up language detection pipeline
def get_lang_detector(nlp, name):
    return LanguageDetector()

nlp = spacy.blank("en")
nlp.add_pipe("sentencizer")
# Add language detector to the pipeline
if not Language.has_factory("language_detector"):
    Language.factory("language_detector", func=get_lang_detector)
nlp.add_pipe("language_detector", last=True)

# regex to identify non-Latin characters (basic Latin + Latin Extended-A/B)
NON_LATIN_REGEX = re.compile(r'[^\u0000-\u024F\u1E00-\u1EFF\s\d' + re.escape(string.punctuation) + ']')

# helper function to calculate Latin character ratio
def get_latin_ratio(text):
    if pd.isna(text) or not str(text).strip(): return 1.0
    text = str(text)
    non_latin = len(NON_LATIN_REGEX.findall(text))
    return 1 - (non_latin / max(len(text), 1))

# audit function for each row
def audit_row(row):
    blurb = str(row.get('blurb', ''))
    story = str(row.get('story_html_text', ''))
    
    # Analyze Blurb
    b_latin = get_latin_ratio(blurb)
    b_doc = nlp(blurb[:500])
    b_lang = b_doc._.language['language']
    b_conf = b_doc._.language['score']
    
    # Analyze Story
    s_latin = get_latin_ratio(story[:2000])
    s_doc = nlp(story[:2000])
    s_lang = s_doc._.language['language']
    s_conf = s_doc._.language['score']
    
    # Determine why this project is flagged as non-English
    reason = []
    if b_latin < 0.90: reason.append("Blurb Low Latin")
    if b_lang != 'en' or b_conf < 0.90: reason.append(f"Blurb AI {b_lang}({b_conf:.2f})")
    if s_latin < 0.90: reason.append("Story Low Latin")
    if s_lang != 'en' or s_conf < 0.80: reason.append(f"Story AI {s_lang}({s_conf:.2f})")
    
    return pd.Series([b_lang, b_conf, b_latin, s_lang, s_conf, s_latin, ", ".join(reason)])

# run the audit
print(f"Auditing {len(non_english_df)} projects...")

# Apply the audit function to each row and store results in new columns
non_english_df[[
    'det_blurb_lang', 'blurb_ai_conf', 'blurb_latin_ratio',
    'det_story_lang', 'story_ai_conf', 'story_latin_ratio', 'rejection_reason'
]] = non_english_df.apply(audit_row, axis=1)

# Save the audit
non_english_df.to_csv('audit_non_english_results.csv', index=False)

Auditing 3874 projects...
Audit complete. Open 'audit_non_english_results.csv' to see the breakdown.


In [ ]:
# Load the audit file
audit_df = pd.read_csv('audit_non_english_results.csv')

print(f"Total projects reviewed: {len(audit_df)}")

# Top Detected Languages (excluding English)
print("Top 5 Non-English Languages Found:")
non_en_counts = audit_df[audit_df['det_story_lang'] != 'en']['det_story_lang'].value_counts().head(5)
print(non_en_counts)

# Rejection Reasons Breakdown
print("Projects by Rejection Reason:")
# Split the reasons and count them
reasons_series = audit_df['rejection_reason'].str.split(', ').explode()
print(reasons_series.value_counts())

# Identify the unclear projects (Potential False Positives)
# These are projects the model says are English, but the Latin Ratio was too low
gray_area = audit_df[
    (audit_df['det_story_lang'] == 'en') & 
    (audit_df['story_ai_conf'] > 0.8) & 
    (audit_df['story_latin_ratio'] < 0.9)
]

print(f"Potential False Positives: {len(gray_area)}")

# Show a few samples from the gray area for manual review
if len(gray_area) > 0:
    print("Samples for manually checking potential false positives:")
    print(gray_area[['slug', 'blurb', 'story_latin_ratio']].head(5))

# save the gray area for manual review
gray_area.to_csv('CHECK_THESE_MANUALLY.csv', index=False)

--- LANGUAGE AUDIT BREAKDOWN ---
Total projects reviewed: 3874

[1] Top 5 Non-English Languages Found:
det_story_lang
es    920
fr    461
de    362
it    189
sv     90
Name: count, dtype: int64

[2] Projects by Rejection Reason:
rejection_reason
Blurb AI es(1.00)       820
Story AI es(1.00)       818
Blurb AI fr(1.00)       414
Story AI fr(1.00)       400
Blurb AI de(1.00)       362
                       ... 
Blurb AI zh-tw(0.86)      1
Blurb AI sw(1.00)         1
Blurb AI ca(0.55)         1
Story AI sv(0.71)         1
Story AI zh-cn(1.00)      1
Name: count, Length: 229, dtype: int64

[3] Potential False Positives (Gray Area): 81
Definition: AI says English, but symbols/formatting triggered a rejection.

--- Samples for Manual Check (The Gray Area) ---
                                                  slug  \
38                    imperfectutopia/imperfect-utopia   
104  1419163893/tradiandlucystory-of-truth-and-cont...   
171      s0021015s/anime-crossroad-chronicle-produce-6   
202

In [ ]:
# Load the gray area file
try:
    check_df = pd.read_csv('CHECK_THESE_MANUALLY.csv')
    
    print(f"Manual Inspection of {len(check_df)} projects ---")
    
    # Selecting columns to inspect
    inspection_view = check_df[['slug', 'blurb', 'story_latin_ratio', 'det_story_lang', 'story_ai_conf', 'urls_web_project']].head(80)
    
    for i, row in inspection_view.iterrows():
        print(f"\n[{i+1}] SLUG: {row['slug']}")
        print(f"    URL: {row['urls_web_project']}")
        print(f"    RATIO: {row['story_latin_ratio']:.2f} | AI: {row['det_story_lang']} ({row['story_ai_conf']:.2f})")
        print(f"    BLURB: {row['blurb']}")
        print("-" * 50)

except FileNotFoundError:
    print("File not found")

--- Manual Inspection of 81 projects ---

[1] SLUG: imperfectutopia/imperfect-utopia
    URL: https://www.kickstarter.com/projects/imperfectutopia/imperfect-utopia
    RATIO: 0.86 | AI: en (1.00)
    BLURB: A contemporary photo book about remote islands of Japan by Toru Morimoto and Tina Bagué
--------------------------------------------------

[2] SLUG: 1419163893/tradiandlucystory-of-truth-and-contradiction
    URL: https://www.kickstarter.com/projects/1419163893/tradiandlucystory-of-truth-and-contradiction
    RATIO: 0.69 | AI: en (0.86)
    BLURB: picture book that creates thoughts
--------------------------------------------------

[3] SLUG: s0021015s/anime-crossroad-chronicle-produce-6
    URL: https://www.kickstarter.com/projects/s0021015s/anime-crossroad-chronicle-produce-6
    RATIO: 0.76 | AI: en (1.00)
    BLURB: Production of the original anime. オリジナルのアニメの制作
My dream is to be made into an anime on TV.テレビアニメ化を目指しています。
--------------------------------------------------

[4] S